# Task 02: PII Detection and Anonymization

Use `knowledgator/gliner-pii-edge-v1.0` to detect and anonymize PII in documents.

You will:
1. Load the model and detect PII in a test document
2. Implement `anonymize_text()` that replaces detected PII with `<LABEL>` placeholders
3. Implement `anonymize_batch()` for efficient multi-document anonymization

## Setup

In [1]:
from gliner import GLiNER
import json, os

fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))
with open(os.path.join(fixtures, "pii_documents.json")) as f:
    pii_documents = json.load(f)

print(f"✓ Loaded {len(pii_documents)} documents")

✓ Loaded 8 documents


## Part 1: Load Model and Detect PII

Load `knowledgator/gliner-pii-edge-v1.0` into `pii_model`.

Then detect PII in the text below using labels `["name", "phone number", "email address"]`
with `threshold=0.3`. Store results in `pii_entities`.

In [2]:
# YOUR CODE HERE
test_text = "John Smith called our support line from 415-555-0182. His email is john.smith@example.com."
pii_labels = ["name", "phone number", "email address"]

# pii_model = ...
# pii_entities = ...


In [3]:
# TEST — model loaded
assert 'pii_model' in dir() or 'pii_model' in vars(), "Load the model into variable 'pii_model'"
assert hasattr(pii_model, 'predict_entities'), "pii_model must have .predict_entities() method"
print("✓ pii_model loaded")

AssertionError: Load the model into variable 'pii_model'

In [4]:
# TEST — PII detected
assert 'pii_entities' in dir() or 'pii_entities' in vars(), "Store results in 'pii_entities'"
assert isinstance(pii_entities, list), "predict_entities returns a list"
assert len(pii_entities) >= 2, (
    f"Expected at least 2 PII entities in test_text, found {len(pii_entities)}. "
    "The text contains a name, phone number, and email address."
)

required_keys = {"text", "label", "start", "end", "score"}
for i, e in enumerate(pii_entities):
    missing = required_keys - e.keys()
    assert not missing, f"Entity {i} missing keys: {missing}"

labels_found = {e["label"] for e in pii_entities}
assert len(labels_found) >= 2, (
    f"Expected at least 2 distinct PII types, found: {labels_found}"
)

print(f"✓ Found {len(pii_entities)} PII entities across {len(labels_found)} types")
for e in pii_entities:
    print(f"  [{e['label']:20}] {e['text']!r:30} score={e['score']:.2f}")

AssertionError: Store results in 'pii_entities'

## Part 2: Implement `anonymize_text`

Implement `anonymize_text(model, text, labels, threshold=0.3) -> str` that:
1. Detects PII entities in `text`
2. Replaces each detected span with `<LABEL_TYPE>` (e.g. `<NAME>`, `<PHONE_NUMBER>`)
3. Returns the anonymized string

**Important:** Replace spans from end to start to preserve offset correctness.

In [5]:
# YOUR CODE HERE
def anonymize_text(model, text, labels, threshold=0.3):
    """
    Replace PII spans in text with <LABEL_TYPE> placeholders.

    Args:
        model: GLiNER PII model
        text: input string
        labels: list of PII label names to detect
        threshold: confidence threshold for detection

    Returns:
        Anonymized string with PII replaced by <LABEL> tags
    """
    pass


In [6]:
# TEST — anonymize_text basic behavior
result = anonymize_text(pii_model, test_text, pii_labels)

assert isinstance(result, str), "anonymize_text must return a string"
assert result != test_text, "Anonymized text must differ from the original"
assert "John Smith" not in result, "Name 'John Smith' should be anonymized"
assert "<" in result and ">" in result, "Result must contain <PLACEHOLDER> tags"

print(f"✓ anonymize_text works correctly")
print(f"  Original:   {test_text}")
print(f"  Anonymized: {result}")

NameError: name 'pii_model' is not defined

In [7]:
# TEST — HR record with more PII types
hr_text = pii_documents[1]["text"]  # Has name, DOB, SSN, address, phone
hr_labels = ["name", "dob", "ssn", "location address", "phone number"]

hr_result = anonymize_text(pii_model, hr_text, hr_labels)

assert isinstance(hr_result, str), "Result must be a string"
assert hr_result != hr_text, "HR document should be anonymized"
# The HR doc contains 'Sarah Johnson' — should be masked
assert "Sarah Johnson" not in hr_result, "Name should be anonymized in HR document"

print(f"✓ HR document anonymized")
print(f"  Original:   {hr_text}")
print(f"  Anonymized: {hr_result}")

NameError: name 'pii_model' is not defined

## Part 3: Implement `anonymize_batch`

Implement `anonymize_batch(model, documents, labels, threshold=0.3) -> list[str]`
that anonymizes a list of documents.

Use `model.inference()` for batch inference instead of calling `predict_entities` in a loop.

In [8]:
# YOUR CODE HERE
def anonymize_batch(model, documents, labels, threshold=0.3):
    """
    Anonymize a list of document strings.

    Args:
        model: GLiNER PII model
        documents: list of raw text strings
        labels: list of PII label names
        threshold: confidence threshold

    Returns:
        List of anonymized strings, same length as documents
    """
    pass


In [9]:
# TEST — anonymize_batch
full_pii_labels = [
    "name", "phone number", "email address", "ssn",
    "credit card", "credit card expiration", "cvv",
    "location address", "dob", "account number",
    "passport number", "driver license", "password", "username",
    "routing number", "bank account", "vehicle id",
]
texts = [doc["text"] for doc in pii_documents]
anonymized = anonymize_batch(pii_model, texts, full_pii_labels)

assert isinstance(anonymized, list), "anonymize_batch must return a list"
assert len(anonymized) == len(pii_documents), (
    f"Expected {len(pii_documents)} results, got {len(anonymized)}"
)
for i, anon in enumerate(anonymized):
    assert isinstance(anon, str), f"Result {i} must be a string"

pii_found = sum(1 for a in anonymized if "<" in a)
assert pii_found >= len(pii_documents) // 2, (
    f"Expected at least half the documents to have PII, but only {pii_found}/{len(pii_documents)} did"
)

print(f"✓ anonymize_batch processed {len(anonymized)} documents")
print(f"  {pii_found}/{len(pii_documents)} documents had PII detected\n")
for i, (orig, anon) in enumerate(zip(texts, anonymized)):
    status = "🔒" if "<" in anon else "  "
    print(f"  {status} Doc {i+1} [{pii_documents[i]['domain']:12}]: {anon[:70]}...")

NameError: name 'pii_model' is not defined